In [1]:
"""
================================================================================
  Architecture Search (NAS) & Efficient Operators — CIFAR-10
  Reference Comment: Mathematical Foundations & Code Structure
================================================================================

OVERVIEW
--------
This script designs and trains TWO new architectures entirely from scratch on
CIFAR-10. The baseline ResNet-50 is a comparison reference ONLY — its weights
are never reused, fine-tuned, or transferred.

  Baseline (reference) : ResNet-50
  Model A              : DARTS NAS  → discrete cell network
  Model B              : MobileNetV1-style depthwise-separable network
  
  
Workflow
--------
1. Baseline already trained: ResNet-50 on CIFAR-10  (loaded for metrics only)
2. Design new model A : DARTS cell search  (pure-PyTorch, no NNI needed)
   - Define a search space of 7 candidate ops per edge
   - Run bilevel DARTS optimisation (arch-alpha + weights W jointly)
   - Extract best ops via argmax(alpha) -> build a fixed discrete network
   - Train discrete network from scratch on CIFAR-10
3. Design new model B : MobileNetV1-style network (all DW-sep ops)
   - Build from scratch, random initialisation
   - Train from scratch on CIFAR-10
4. Compare BOTH new models vs. baseline ResNet-50 on all 8 metrics

================================================================================
PART 1 — DARTS SEARCH SPACE (MixedOp)
================================================================================
Each edge (i→j) in the cell graph computes a SOFT mixture of K candidate ops:

       K-1
  ō =  Σ   softmax(α)_k · o_k(x)
       k=0

  where  α ∈ ℝᴷ  are learnable architecture parameters (one vector per edge),
  K = 7 candidate operations:
    { skip, avg_pool_3×3, max_pool_3×3, sep_conv_3×3,
      dil_conv_3×3, conv_3×3, conv_1×1 }

After search:  ô_best = o_{argmax_k α_k}(x)   (discrete / hard selection)

================================================================================
PART 2 — DARTS CELL (DARTSCell / DiscreteCell)
================================================================================
A cell has 2 external inputs (s₀, s₁) and N=4 intermediate nodes.
Node i aggregates outputs from ALL previous nodes j < i+2:

  n_i =  Σ_{j=0}^{i+1}  ō_{i,j}(n_j)       (search phase — soft)
  n_i =  Σ_{j=0}^{i+1}  o_{i,j}^*(n_j)      (discrete phase — hard)

Cell output = concatenation of all intermediate node outputs:
  cell_out = Concat(n_2, n_3, …, n_{N+1})  ∈ ℝ^{N·C × H × W}

A 1×1 projection then maps back to C channels:
  s_out = Conv_{1×1}(cell_out)  ∈ ℝ^{C × H × W}

================================================================================
PART 3 — DARTS BILEVEL OPTIMISATION (run_darts_search)
================================================================================
DARTS solves a bilevel problem with two parameter sets:
  α = architecture parameters    (alphas_normal, alphas_reduction)
  W = network weight parameters

Objective (shared cross-entropy loss ℒ):

  min_α  ℒ_val  ( W*(α), α )
  s.t.   W*(α) = argmin_W  ℒ_train(W, α)

Approximated by alternating one-step updates each mini-batch:

  Step 1 (arch)   :  α ← α − η_α · ∇_α ℒ(x_val,   y_val;   W, α)
  Step 2 (weights):  W ← W − η_W · ∇_W ℒ(x_train, y_train; W, α)

  η_α = 3×10⁻⁴  (Adam),   η_W = 3×10⁻⁴  (Adam),   λ_arch = 1×10⁻³

================================================================================
PART 4 — DARTS NETWORK STRUCTURE (DARTSSearchNet / DARTSDiscreteNet)
================================================================================
  Stem   : Conv_{3×3}(3 → C)
  Cells  : num_cells=8 stacked cells
             ∟ 2 reduction cells at depths ⌊L/3⌋ and ⌊2L/3⌋ (stride-2)
             ∟ remaining cells are normal (stride-1)
  Output : GlobalAvgPool → FC(C → 10)

Spatial resolution schedule (32×32 input, C=16):
  After reduction cell 1:  16×16
  After reduction cell 2:   8×8

================================================================================
PART 5 — DEPTHWISE-SEPARABLE CONVOLUTION (DWSepBlock / SepConv)
================================================================================
Standard convolution FLOPs (K=3, input C_in, output C_out, spatial H×W):

  FLOPs_std = 2 · K² · C_in · C_out · H · W

Depthwise-Separable factorisation:
  Depthwise  (K×K, groups=C_in) :  2 · K² · C_in       · H · W
  Pointwise  (1×1)              :  2 · C_in · C_out     · H · W
  ─────────────────────────────────────────────────────────────
  FLOPs_dw_sep = 2 · (K² · C_in + C_in · C_out) · H · W

Reduction ratio:
  FLOPs_dw_sep / FLOPs_std = 1/C_out + 1/K²  ≈ 1/9  for K=3, C_out≫1
  → ~8–9× fewer multiply-adds for a 3×3 conv

================================================================================
PART 6 — MOBILENETV1 ARCHITECTURE (MobileNetV1CIFAR)
================================================================================
  Stem     : Conv_{3×3}(3 → 32·α_w),  stride=1  (preserves 32×32)
  Backbone : 12 DWSepBlocks with channel/stride schedule:
               64/1 → 128/2 → 128/1 → 256/2 → 256/1 → 512/2
               → 512/1 ×5 → 1024/1
  Output   : GlobalAvgPool → FC(1024·α_w → 10)

  Width multiplier α_w ∈ {0.5, 1.0} uniformly scales all channel counts:
    C_i' = ⌊α_w · C_i⌋

  Parameter count scales as  O(α_w²) ,  FLOPs scale as  O(α_w²)

================================================================================
PART 7 — FLOPs COUNTER (count_flops_and_params)
================================================================================
FLOPs are estimated as 2×MACs via forward hooks:

  Conv2d :  FLOPs = 2 · (C_in/groups) · K_H · K_W · C_out · H_out · W_out · B
  Linear :  FLOPs = 2 · B · in_features · out_features

  Total FLOPs reported = sum over all Conv2d and Linear layers in one forward pass
  with batch size B=1  (input 1 × 3 × 32 × 32)

================================================================================
PART 8 — TRAINING FROM SCRATCH (train_from_scratch)
================================================================================
Both discrete DARTS net and MobileNetV1 are trained identically:

  Optimiser : SGD with Nesterov momentum
    W_{t+1} = W_t − η · (∇ℒ + λ·W_t)   with momentum μ=0.9, λ=3×10⁻⁴

  LR schedule : Cosine annealing
    η_t = η_min + ½(η_max − η_min)(1 + cos(πt/T))
    η_max = 0.025,  η_min = 0,  T = 100 epochs

  Loss : Cross-Entropy
    ℒ = −(1/N) Σ_i log p_{y_i}

================================================================================
PART 9 — EVALUATION METRICS
================================================================================
All models evaluated on the CIFAR-10 test set (10 000 samples):

  Accuracy   = (TP + TN) / N_total
  Precision  = (1/C) Σ_c  TP_c / (TP_c + FP_c)      (macro average)
  Recall     = (1/C) Σ_c  TP_c / (TP_c + FN_c)      (macro average)
  F1         = (1/C) Σ_c  2·P_c·R_c / (P_c + R_c)   (macro average)

  Params (M)      = Σ_l |θ_l| / 10⁶
  Size disk (MB)  = sizeof(state_dict) on disk in MiB
  FLOPs (M)       = total FLOPs (Part 7) / 10⁶
  GPU latency     = mean wall-time over 100 forward passes (CUDA events), ms
  CPU latency     = mean wall-time over 20 forward passes (perf_counter),  ms

================================================================================
OUTPUT
--------
  __1__NAS_&_efficient_operators.json  — full metrics + architecture details
================================================================================
"""


import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, accuracy_score)

warnings.filterwarnings("ignore")

# for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__1_v2_NAS_&_efficient_operators.json"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
NUM_WORKERS = 2
NUM_CLASSES = 10
INFERENCE_RUNS = 100

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

# ── DARTS search hyperparams ───────────────────────────────────────────────────
DARTS_C         = 16    # base channels in search / discrete network
DARTS_NUM_CELLS = 8     # number of stacked cells
DARTS_NUM_NODES = 4     # intermediate nodes per cell
SEARCH_EPOCHS   = 30    # joint (alpha + W) search epochs
SEARCH_SUBSET   = 10_000
LR_WEIGHTS      = 3e-4
LR_ARCH         = 3e-4
ARCH_WD         = 1e-3

# ── Training from scratch (shared by discrete net + MobileNet) ─────────────────
TRAIN_EPOCHS = 100
LR_TRAIN     = 0.025
MOMENTUM     = 0.9
WEIGHT_DECAY = 3e-4

# ── DARTS search space (7 candidate ops per edge) ─────────────────────────────
OPS_NAMES = [
    "skip",
    "avg_pool_3x3",
    "max_pool_3x3",
    "sep_conv_3x3",
    "dil_conv_3x3",
    "conv_3x3",
    "conv_1x1",
]
NUM_OPS = len(OPS_NAMES)

In [2]:
# =============================================================================
# Search-space operations
# =============================================================================

class SepConv(nn.Module):
    """Depthwise-separable 3x3 applied twice (standard DARTS cell op)."""
    def __init__(self, C, stride):
        super().__init__()
        self.op = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=stride, padding=1, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Conv2d(C, C, 3, stride=1, padding=1, groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)


class DilSepConv(nn.Module):
    """Dilated depthwise-separable 3x3 (dilation=2) applied twice."""
    def __init__(self, C, stride):
        super().__init__()
        self.op = nn.Sequential(
            nn.Conv2d(C, C, 3, stride=stride, padding=2, dilation=2,
                      groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True),
            nn.Conv2d(C, C, 3, stride=1, padding=2, dilation=2,
                      groups=C, bias=False),
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C),
        )
    def forward(self, x):
        return self.op(x)


def make_op(name, C, stride):
    """Instantiate a candidate operation by name."""
    if name == "skip":
        return (nn.Identity() if stride == 1
                else nn.Sequential(nn.AvgPool2d(2, 2),
                                   nn.Conv2d(C, C, 1, bias=False),
                                   nn.BatchNorm2d(C)))
    elif name == "avg_pool_3x3":
        return nn.AvgPool2d(3, stride=stride, padding=1)
    elif name == "max_pool_3x3":
        return nn.MaxPool2d(3, stride=stride, padding=1)
    elif name == "sep_conv_3x3":
        return SepConv(C, stride)
    elif name == "dil_conv_3x3":
        return DilSepConv(C, stride)
    elif name == "conv_3x3":
        return nn.Sequential(
            nn.Conv2d(C, C, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))
    elif name == "conv_1x1":
        return nn.Sequential(
            nn.Conv2d(C, C, 1, stride=stride, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))
    else:
        raise ValueError(f"Unknown op: {name}")


In [3]:
# =============================================================================
# DARTS: Search network
# =============================================================================

class MixedOp(nn.Module):
    """
    DARTS mixed operation: differentiable weighted sum of all candidate ops.
    Weights (alpha) are passed in from outside so one alpha vector is shared
    across all cells of the same type (normal / reduction).
    """
    def __init__(self, C, stride):
        super().__init__()
        self._ops = nn.ModuleList(
            [make_op(name, C, stride) for name in OPS_NAMES])

    def forward(self, x, weights):
        return sum(w * op(x) for w, op in zip(weights, self._ops))


class DARTSCell(nn.Module):
    """
    Standard DARTS cell: DARTS_NUM_NODES intermediate nodes.
    Each node receives inputs from all previous nodes plus the 2 external inputs.
    Output = concatenation of all DARTS_NUM_NODES node outputs.

    reduction_prev: if True, preprocess0 downsamples s0 to match s1's spatial.
    """
    def __init__(self, C, reduction, reduction_prev):
        super().__init__()
        self.reduction = reduction
        stride = 2 if reduction else 1

        self.mixed_ops = nn.ModuleList()
        self.edge_map  = []
        for i in range(DARTS_NUM_NODES):
            for j in range(i + 2):
                s = stride if j < 2 else 1
                self.mixed_ops.append(MixedOp(C, s))
                self.edge_map.append((i, j))

        if reduction_prev:
            self.preprocess0 = nn.Sequential(
                nn.AvgPool2d(2, 2, count_include_pad=False),
                nn.Conv2d(C, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True))
        else:
            self.preprocess0 = nn.Sequential(
                nn.Conv2d(C, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True))

        self.preprocess1 = nn.Sequential(
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))

    def forward(self, s0, s1, arch_alphas):
        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)
        states = [s0, s1]
        op_idx = 0
        for i in range(DARTS_NUM_NODES):
            node_outs = []
            for j in range(i + 2):
                w = F.softmax(arch_alphas[op_idx], dim=0)
                node_outs.append(self.mixed_ops[op_idx](states[j], w))
                op_idx += 1
            states.append(sum(node_outs))
        return torch.cat(states[2:], dim=1)


class DARTSSearchNet(nn.Module):
    """
    Full DARTS search network for CIFAR-10.

    Structure:
      Stem (3 -> C, 3x3 conv) ->
      num_cells stacked DARTSCells (2 reduction cells at 1/3 and 2/3 depth) ->
      Global Average Pool -> FC(10)

    Architecture parameters (alpha) are shared across same-type cells.
    """
    def __init__(self, C=DARTS_C, num_cells=DARTS_NUM_CELLS,
                 num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))

        self.cells      = nn.ModuleList()
        self.cell_types = []
        self.proj       = nn.ModuleList()
        reduction_prev  = False

        for i in range(num_cells):
            reduction = (i in [num_cells // 3, 2 * num_cells // 3])
            self.cells.append(
                DARTSCell(C, reduction=reduction, reduction_prev=reduction_prev))
            self.cell_types.append("reduction" if reduction else "normal")
            self.proj.append(nn.Sequential(
                nn.Conv2d(C * DARTS_NUM_NODES, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True)))
            reduction_prev = reduction

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(C, num_classes)

        n_edges = len(self.cells[0].edge_map)
        self.alphas_normal = nn.ParameterList(
            [nn.Parameter(torch.zeros(NUM_OPS)) for _ in range(n_edges)])
        self.alphas_reduction = nn.ParameterList(
            [nn.Parameter(torch.zeros(NUM_OPS)) for _ in range(n_edges)])

    def forward(self, x):
        s0 = s1 = self.stem(x)
        for cell, ctype, proj in zip(self.cells, self.cell_types, self.proj):
            alphas = (self.alphas_normal
                      if ctype == "normal" else self.alphas_reduction)
            s0, s1 = s1, proj(cell(s0, s1, list(alphas)))
        return self.fc(self.gap(s1).flatten(1))

    def arch_parameters(self):
        return list(self.alphas_normal) + list(self.alphas_reduction)

    def network_parameters(self):
        arch_ids = {id(p) for p in self.arch_parameters()}
        return [p for p in self.parameters() if id(p) not in arch_ids]

    def export_arch(self):
        """Return per-edge best op names for normal and reduction cells."""
        def best(plist):
            return [OPS_NAMES[p.argmax().item()] for p in plist]
        return {"normal": best(self.alphas_normal),
                "reduction": best(self.alphas_reduction)}

    def export_probs(self):
        """Return softmax probabilities for every edge."""
        def probs(plist):
            return [{OPS_NAMES[k]: round(v, 4)
                     for k, v in enumerate(F.softmax(p, dim=0).tolist())}
                    for p in plist]
        return {"normal": probs(self.alphas_normal),
                "reduction": probs(self.alphas_reduction)}


In [4]:
# =============================================================================
# DARTS: Discrete network (fixed ops after search)
# =============================================================================

class DiscreteCell(nn.Module):
    """Cell with FIXED operations from DARTS argmax(alpha)."""
    def __init__(self, C, reduction, reduction_prev, normal_ops, reduction_ops):
        super().__init__()
        self.reduction = reduction
        ops_list = reduction_ops if reduction else normal_ops
        stride   = 2 if reduction else 1

        self.fixed_ops = nn.ModuleList()
        self.edge_map  = []
        idx = 0
        for i in range(DARTS_NUM_NODES):
            for j in range(i + 2):
                s = stride if j < 2 else 1
                self.fixed_ops.append(make_op(ops_list[idx], C, s))
                self.edge_map.append((i, j))
                idx += 1

        if reduction_prev:
            self.preprocess0 = nn.Sequential(
                nn.AvgPool2d(2, 2, count_include_pad=False),
                nn.Conv2d(C, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True))
        else:
            self.preprocess0 = nn.Sequential(
                nn.Conv2d(C, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True))

        self.preprocess1 = nn.Sequential(
            nn.Conv2d(C, C, 1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))

    def forward(self, s0, s1):
        s0 = self.preprocess0(s0)
        s1 = self.preprocess1(s1)
        states = [s0, s1]
        op_idx = 0
        for i in range(DARTS_NUM_NODES):
            node_outs = [self.fixed_ops[op_idx + j](states[j])
                         for j in range(i + 2)]
            op_idx += i + 2
            states.append(sum(node_outs))
        return torch.cat(states[2:], dim=1)


class DARTSDiscreteNet(nn.Module):
    """
    Fixed-architecture network built from DARTS search results.
    Trained entirely from scratch (random initialisation).
    """
    def __init__(self, C=DARTS_C, num_cells=DARTS_NUM_CELLS,
                 num_classes=NUM_CLASSES, normal_ops=None, reduction_ops=None):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, C, 3, padding=1, bias=False),
            nn.BatchNorm2d(C), nn.ReLU(inplace=True))

        self.cells = nn.ModuleList()
        self.proj  = nn.ModuleList()
        reduction_prev = False
        for i in range(num_cells):
            reduction = (i in [num_cells // 3, 2 * num_cells // 3])
            self.cells.append(DiscreteCell(
                C, reduction, reduction_prev, normal_ops, reduction_ops))
            self.proj.append(nn.Sequential(
                nn.Conv2d(C * DARTS_NUM_NODES, C, 1, bias=False),
                nn.BatchNorm2d(C), nn.ReLU(inplace=True)))
            reduction_prev = reduction

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Linear(C, num_classes)

    def forward(self, x):
        s0 = s1 = self.stem(x)
        for cell, proj in zip(self.cells, self.proj):
            s0, s1 = s1, proj(cell(s0, s1))
        return self.fc(self.gap(s1).flatten(1))


In [5]:
# =============================================================================
# Efficient Operators: MobileNetV1 from scratch
# =============================================================================

class DWSepBlock(nn.Module):
    """
    Depthwise-Separable Convolution block (MobileNet V1 building block).

    Complexity reduction vs standard conv (K=3):
      Standard : K2 * C_in * C_out * H * W
      DW-Sep   : (K2 * C_in + C_in * C_out) * H * W  =>  ~8-9x fewer FLOPs
    """
    def __init__(self, C_in, C_out, stride=1):
        super().__init__()
        self.dw    = nn.Conv2d(C_in, C_in, 3, stride=stride, padding=1,
                               groups=C_in, bias=False)
        self.bn_dw = nn.BatchNorm2d(C_in)
        self.pw    = nn.Conv2d(C_in, C_out, 1, bias=False)
        self.bn_pw = nn.BatchNorm2d(C_out)

    def forward(self, x):
        x = F.relu(self.bn_dw(self.dw(x)), inplace=True)
        return F.relu(self.bn_pw(self.pw(x)), inplace=True)


class MobileNetV1CIFAR(nn.Module):
    """
    MobileNet V1-style network built entirely from scratch for CIFAR-10.
    Every spatial convolution is a Depthwise-Separable block.
    No ResNet weights, no pretrained features -- pure from-scratch design.
    width_mult scales all channel counts uniformly.
    """
    def __init__(self, width_mult=1.0, num_classes=NUM_CLASSES):
        super().__init__()

        def c(n):
            return max(1, int(n * width_mult))

        # CIFAR-10: 32x32 input -- stride-1 stem keeps spatial resolution
        self.stem = nn.Sequential(
            nn.Conv2d(3, c(32), 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(c(32)), nn.ReLU(inplace=True))

        cfg = [
            (c(64),  1), (c(128), 2), (c(128), 1),
            (c(256), 2), (c(256), 1), (c(512), 2),
            (c(512), 1), (c(512), 1), (c(512), 1),
            (c(512), 1), (c(512), 1), (c(1024), 1),
        ]
        layers = []
        ch = c(32)
        for C_out, s in cfg:
            layers.append(DWSepBlock(ch, C_out, stride=s))
            ch = C_out

        self.features = nn.Sequential(*layers)
        self.gap      = nn.AdaptiveAvgPool2d(1)
        self.fc       = nn.Linear(ch, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.features(x)
        return self.fc(self.gap(x).flatten(1))


In [6]:
# =============================================================================
# Baseline reference (never modified)
# =============================================================================

def build_resnet50_baseline(num_classes=NUM_CLASSES):
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


# =============================================================================
# Data
# =============================================================================

def get_train_loader(subset=None):
    tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        "../data", train=True, download=True, transform=tf)
    if subset:
        idx = torch.randperm(len(ds))[:subset].tolist()
        ds  = Subset(ds, idx)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)


def get_test_loader():
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(
        "../data", train=False, download=True, transform=tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)


In [7]:
# =============================================================================
# Metrics helpers (same contract as pruning / KD scripts)
# =============================================================================

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro",
                                           zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro",
                                        zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro",
                                    zero_division=0)),
    }


def param_count(model):
    return sum(p.numel() for p in model.parameters())

def measure_inference(model: nn.Module, device, batch_size=128, runs=50) -> dict:
    """
    Returns a dict with per-image (single), batch, and throughput timings.
    Uses CUDA events on GPU for highest accuracy; falls back to perf_counter on CPU.
    """
    m = model.eval().to(device)
    use_cuda = device.type == "cuda"

    def _timed_run(inp: torch.Tensor, n: int) -> float:
        """Returns total elapsed seconds for n forward passes."""
        with torch.no_grad():
            # warm-up
            for _ in range(3):
                m(inp)
            if use_cuda:
                torch.cuda.synchronize()
                start_evt = torch.cuda.Event(enable_timing=True)
                end_evt   = torch.cuda.Event(enable_timing=True)
                start_evt.record()
                for _ in range(n):
                    m(inp)
                end_evt.record()
                torch.cuda.synchronize()
                return start_evt.elapsed_time(end_evt) / 1000.0  # ms → s
            else:
                t0 = time.perf_counter()
                for _ in range(n):
                    m(inp)
                return time.perf_counter() - t0

    # Single-image latency
    single = torch.randn(1, 3, 32, 32, device=device)
    single_s = _timed_run(single, runs)
    single_ms = single_s / runs * 1000

    # Batch latency
    batch = torch.randn(batch_size, 3, 32, 32, device=device)
    batch_s = _timed_run(batch, runs)
    batch_ms = batch_s / runs * 1000

    per_img_ms  = batch_ms / batch_size
    throughput  = batch_size * runs / batch_s   # images / second

    timing_method = (
        "CUDA events + torch.cuda.synchronize()" if use_cuda
        else "time.perf_counter() (CPU)"
    )

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms,  4),
        "per_img_gpu_ms"     : round(per_img_ms, 6),
        "throughput_imgs_sec": round(throughput, 2),
        "timing_method"      : timing_method,
    }

def compute_flops(model: nn.Module, device, input_size=(1, 3, 32, 32)) -> float:
    """
    Estimates FLOPs (as GFLOPs) by registering forward hooks on Conv2d and Linear layers.
    Counts multiply-accumulate operations × 2 (one mul + one add = 2 FLOPs).
    Returns GFLOPs (float).
    """
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []

    def conv_hook(module, inp, out):
        # out: (N, C_out, H_out, W_out)
        N, C_out, H_out, W_out = out.shape
        C_in  = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size, module.kernel_size)
        groups = module.groups
        # MACs per output element = (C_in / groups) * kH * kW
        macs = N * C_out * H_out * W_out * (C_in // groups) * kH * kW
        total_flops[0] += 2 * macs

    def linear_hook(module, inp, out):
        # out: (N, C_out)
        in_f  = module.in_features
        out_f = module.out_features
        N     = inp[0].shape[0]
        total_flops[0] += 2 * N * in_f * out_f

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))

    dummy = torch.randn(*input_size, device=device)
    with torch.no_grad():
        m(dummy)

    for h in hooks:
        h.remove()

    return round(total_flops[0] / 1e9, 6)  # GFLOPs

def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def collect_metrics(model, loader, baseline, label):
    print(f"\n  Collecting metrics for [{label}]...")
    perf     = evaluate(model, loader)
    params   = param_count(model)
    flops_g  = compute_flops(model, DEVICE)
    size_disk = disk_size_mb(model)
    infer    = measure_inference(model, DEVICE)
    acc_drop = baseline["accuracy"] - perf["accuracy"]

    print(f"    Acc={perf['accuracy']:.4f}  (drop={acc_drop:+.4f})  "
          f"F1={perf['f1']:.4f}  Params={params/1e6:.3f}M  "
          f"FLOPs={flops_g:.4f}G  "
          f"Single={infer['single_img_gpu_ms']:.2f}ms  "
          f"Batch={infer['batch128_gpu_ms']:.2f}ms  "
          f"Disk={size_disk:.1f}MB")

    return {
        "label"            : label,
        "accuracy"         : round(perf["accuracy"],  6),
        "precision"        : round(perf["precision"], 6),
        "recall"           : round(perf["recall"],    6),
        "f1"               : round(perf["f1"],        6),
        "accuracy_drop"    : round(acc_drop, 6),
        "params"           : params,
        "params_M"         : round(params / 1e6, 4),
        "flops_G"          : flops_g,
        "size_disk_mb"     : size_disk,
        "inference_ms"     : infer,
    }


In [8]:
# =============================================================================
# Training from scratch
# =============================================================================

def train_from_scratch(model, train_loader, epochs, label):
    """SGD + cosine annealing, random initialisation (no pretrained weights)."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR_TRAIN,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

    print(f"\n  Training [{label}] from scratch: {epochs} epochs, lr={LR_TRAIN}")

    for ep in range(1, epochs + 1):
        model.train()
        correct = total = 0
        run_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out  = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            correct  += (out.argmax(1) == y).sum().item()
            total    += y.size(0)
            run_loss += loss.item() * y.size(0)
        scheduler.step()
        if ep % 20 == 0 or ep == 1:
            print(f"    ep {ep:3d}/{epochs} | "
                  f"loss={run_loss/total:.4f} | acc={correct/total:.4f}")
    return model


In [9]:
# =============================================================================
# DARTS bilevel search
# =============================================================================

def run_darts_search(search_loader):
    """
    Bilevel DARTS: alternates arch-step (update alpha on val half)
    and weight-step (update W on train half) each mini-batch.

    Returns: (normal_ops, reduction_ops, alpha_probs)
    """
    net = DARTSSearchNet(
        C=DARTS_C, num_cells=DARTS_NUM_CELLS,
        num_classes=NUM_CLASSES).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    w_opt = torch.optim.Adam(net.network_parameters(),
                             lr=LR_WEIGHTS, weight_decay=WEIGHT_DECAY)
    a_opt = torch.optim.Adam(net.arch_parameters(),
                             lr=LR_ARCH, weight_decay=ARCH_WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(w_opt, SEARCH_EPOCHS)

    n_edges = len(net.cells[0].edge_map)
    print(f"\n  DARTS search: {SEARCH_EPOCHS} epochs | "
          f"{NUM_OPS} ops x {n_edges} edges | C={DARTS_C} cells={DARTS_NUM_CELLS}")

    for ep in range(1, SEARCH_EPOCHS + 1):
        net.train()
        total_loss = n_batches = 0

        for x, y in search_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            B    = x.size(0)
            half = B // 2
            if half == 0:
                continue

            # Arch step: update alpha on second half
            a_opt.zero_grad()
            criterion(net(x[half:]), y[half:]).backward()
            a_opt.step()

            # Weight step: update W on first half
            w_opt.zero_grad()
            loss_w = criterion(net(x[:half]), y[:half])
            loss_w.backward()
            w_opt.step()

            total_loss += loss_w.item()
            n_batches  += 1

        scheduler.step()

        if ep % 10 == 0 or ep == 1:
            arch = net.export_arch()
            print(f"    ep {ep:2d}/{SEARCH_EPOCHS} | "
                  f"loss={total_loss/n_batches:.4f} | "
                  f"normal[:3]={arch['normal'][:3]}")

    arch  = net.export_arch()
    probs = net.export_probs()
    print(f"\n  Search complete.")
    print(f"  Normal    ops: {arch['normal']}")
    print(f"  Reduction ops: {arch['reduction']}")

    del net
    return arch["normal"], arch["reduction"], probs


In [10]:
# =============================================================================
# Main
# =============================================================================

def main():
    print(f"\n{'='*65}")
    print("  Architecture Search (NAS) & Efficient Operators -- CIFAR-10")
    print(f"  Approach: NEW architectures designed and trained FROM SCRATCH")
    print(f"  Device  : {DEVICE}")
    print(f"{'='*65}\n")

    # Load baseline metrics (comparison reference, weights NOT reused)
    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    print(f"  Baseline ResNet-50 accuracy: {baseline['accuracy']:.4f}")

    base_model = build_resnet50_baseline().to(DEVICE)
    base_model.load_state_dict(
        torch.load(BASELINE_MODEL_PATH, map_location=DEVICE))
    base_params = param_count(base_model)
    base_flops  = compute_flops(base_model, DEVICE)   # GFLOPs
    base_disk   = disk_size_mb(base_model)
    print(f"  Baseline params  : {base_params/1e6:.2f}M")
    print(f"  Baseline FLOPs   : {base_flops:.4f} G")
    del base_model

    search_loader = get_train_loader(subset=SEARCH_SUBSET)
    train_loader  = get_train_loader()
    test_loader   = get_test_loader()

    results = {}

    # =========================================================================
    # A) DARTS Neural Architecture Search
    # =========================================================================
    print(f"\n{'─'*55}")
    print("  A) DARTS NAS -- design new architecture from scratch")
    print(f"{'─'*55}")
    print(f"  Search space: {OPS_NAMES}")

    t0 = time.perf_counter()

    normal_ops, reduction_ops, alpha_probs = run_darts_search(search_loader)

    darts_net = DARTSDiscreteNet(
        C=DARTS_C, num_cells=DARTS_NUM_CELLS, num_classes=NUM_CLASSES,
        normal_ops=normal_ops, reduction_ops=reduction_ops)
    darts_params = sum(p.numel() for p in darts_net.parameters())
    print(f"\n  Discrete DARTS net: {darts_params/1e6:.3f}M params (random init)")

    darts_net = train_from_scratch(
        darts_net, train_loader, TRAIN_EPOCHS, "DARTS-discrete")

    darts_time = round(time.perf_counter() - t0, 1)

    m_darts = collect_metrics(darts_net, test_loader, baseline, "DARTS NAS")
    m_darts.update({
        "search_epochs"         : SEARCH_EPOCHS,
        "train_epochs"          : TRAIN_EPOCHS,
        "search_C"              : DARTS_C,
        "search_num_cells"      : DARTS_NUM_CELLS,
        "search_num_nodes"      : DARTS_NUM_NODES,
        "search_space"          : OPS_NAMES,
        "searched_normal_ops"   : normal_ops,
        "searched_reduction_ops": reduction_ops,
        "alpha_probs"           : alpha_probs,
        "total_time_s"          : darts_time,
    })
    results["darts_nas"] = m_darts
    del darts_net

    # =========================================================================
    # B) Efficient Operators -- MobileNetV1 from scratch
    # =========================================================================
    print(f"\n{'─'*55}")
    print("  B) Efficient Operators -- MobileNetV1, from scratch")
    print(f"{'─'*55}")

    for wm, wm_key in [(0.5, "mobilenet_wm05"), (1.0, "mobilenet_wm10")]:
        label = f"MobileNetV1 wm={wm}"
        t0    = time.perf_counter()

        mob = MobileNetV1CIFAR(width_mult=wm, num_classes=NUM_CLASSES)
        mob_params = sum(p.numel() for p in mob.parameters())
        print(f"\n  Building {label}: {mob_params/1e6:.3f}M params")

        mob      = train_from_scratch(mob, train_loader, TRAIN_EPOCHS, label)
        mob_time = round(time.perf_counter() - t0, 1)

        m = collect_metrics(mob, test_loader, baseline, label)
        m.update({
            "train_epochs" : TRAIN_EPOCHS,
            "width_mult"   : wm,
            "total_time_s" : mob_time,
            "design"       : "All DW-Sep convolutions, built from scratch",
            "flops_formula": "(K2*C_in + C_in*C_out)*H*W per block",
        })
        results[wm_key] = m
        del mob

    # =========================================================================
    # Build JSON report
    # =========================================================================
    def vs_baseline(m):
        return {
            "accuracy_drop"      : round(baseline["accuracy"] - m["accuracy"], 6),
            "params_reduction"   : round(1 - m["params"] / base_params, 4),
            "flops_reduction"    : round(1 - m["flops_G"] / base_flops,  4),
            "disk_size_reduction": round(1 - m["size_disk_mb"] / base_disk, 4),
        }

    report = {
        "method"       : "nas_efficient_operators",
        "description"  : (
            "DARTS cell-based NAS (pure PyTorch, no NNI) + "
            "MobileNetV1-style Depthwise-Separable network. "
            "Both designed and trained FROM SCRATCH on CIFAR-10. "
            "Baseline ResNet-50 is the comparison reference only."
        ),
        "approach"     : "design_from_scratch",
        "key_principle": (
            "Unlike pruning / quantization / distillation which compress an "
            "existing model, NAS and Efficient Operators design new inherently "
            "efficient architectures from the ground up. The baseline ResNet-50 "
            "is NEVER modified or reused here."
        ),
        "baseline": {
            **baseline,
            "params"      : base_params,
            "params_M"    : round(base_params / 1e6, 4),
            "flops_G"     : base_flops,
            "size_disk_mb": base_disk,
        },
        "hyperparams": {
            "darts": {
                "C"             : DARTS_C,
                "num_cells"     : DARTS_NUM_CELLS,
                "num_nodes"     : DARTS_NUM_NODES,
                "search_epochs" : SEARCH_EPOCHS,
                "search_subset" : SEARCH_SUBSET,
                "lr_weights"    : LR_WEIGHTS,
                "lr_arch"       : LR_ARCH,
                "arch_wd"       : ARCH_WD,
            },
            "training": {
                "epochs"       : TRAIN_EPOCHS,
                "lr"           : LR_TRAIN,
                "momentum"     : MOMENTUM,
                "weight_decay" : WEIGHT_DECAY,
                "scheduler"    : "cosine_annealing",
                "init"         : "random (from scratch -- zero baseline weights)",
            },
        },
        "results"               : results,
        "comparison_vs_baseline": {k: vs_baseline(v) for k, v in results.items()},
        "notes": {
            "nas"         : "Pure-PyTorch DARTS bilevel search. No NNI dependency.",
            "search_space": f"{NUM_OPS} ops x {DARTS_NUM_NODES} nodes x {DARTS_NUM_CELLS} cells",
            "mobilenet"   : "MobileNetV1 DW-Sep blocks, width_mult=0.5 and 1.0 variants.",
            "flops"       : "FLOPs = 2 x MACs via forward hooks on Conv2d + Linear.",
            "from_scratch": "No baseline weights used. All models randomly initialised.",
            "dw_sep_complexity": (
                "Standard conv: K2*C_in*C_out*H*W. "
                "DW-Sep: (K2*C_in + C_in*C_out)*H*W. "
                "Ratio: 1/C_out + 1/K2 ~ 8-9x fewer for K=3."
            ),
        },
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)
    print(f"\n  Saved -> {OUTPUT_JSON}")

    # Summary table
    W = 82
    print(f"\n{'='*W}")
    print(f"  {'Model':<28} {'Acc':>7} {'F1':>7} "
          f"{'Params(M)':>10} {'FLOPs(G)':>10} {'Disk(MB)':>9} {'Batch(ms)':>10}")
    print(f"  {'─'*28} {'─'*7} {'─'*7} {'─'*10} {'─'*10} {'─'*9} {'─'*10}")
    print(f"  {'ResNet-50 baseline':<28} "
          f"{baseline['accuracy']:>7.4f} {baseline.get('f1',0):>7.4f} "
          f"{base_params/1e6:>10.2f} {base_flops:>10.4f} "
          f"{base_disk:>9.1f} {'n/a':>10}")
    for v in results.values():
        print(f"  {v['label']:<28} "
              f"{v['accuracy']:>7.4f} {v['f1']:>7.4f} "
              f"{v['params_M']:>10.3f} {v['flops_G']:>10.4f} "
              f"{v['size_disk_mb']:>9.1f} "
              f"{v['inference_ms']['batch128_gpu_ms']:>8.1f}")
    print(f"{'='*W}\n")


if __name__ == "__main__":
    main()


  Architecture Search (NAS) & Efficient Operators -- CIFAR-10
  Approach: NEW architectures designed and trained FROM SCRATCH
  Device  : cuda

  Baseline ResNet-50 accuracy: 0.9320
  Baseline params  : 23.52M
  Baseline FLOPs   : 2.5957 G

───────────────────────────────────────────────────────
  A) DARTS NAS -- design new architecture from scratch
───────────────────────────────────────────────────────
  Search space: ['skip', 'avg_pool_3x3', 'max_pool_3x3', 'sep_conv_3x3', 'dil_conv_3x3', 'conv_3x3', 'conv_1x1']

  DARTS search: 30 epochs | 7 ops x 14 edges | C=16 cells=8
    ep  1/30 | loss=2.2304 | normal[:3]=['conv_1x1', 'max_pool_3x3', 'conv_3x3']
    ep 10/30 | loss=1.5484 | normal[:3]=['max_pool_3x3', 'conv_3x3', 'dil_conv_3x3']
    ep 20/30 | loss=1.2663 | normal[:3]=['conv_3x3', 'conv_3x3', 'dil_conv_3x3']
    ep 30/30 | loss=1.2006 | normal[:3]=['conv_3x3', 'conv_3x3', 'conv_3x3']

  Search complete.
  Normal    ops: ['conv_3x3', 'conv_3x3', 'conv_3x3', 'conv_3x3', 'dil_co